# 📗 인덱스·제약조건·추천 랭킹

앞 시간에는 **무엇을 세고 어떻게 요약하는지**를 배웠습니다. 이번 시간에는 그 조회를 **빠르게** 만들고(인덱스), 데이터가 **망가지지 않게** 지키고(제약조건), 집계를 **추천·랭킹**으로 잇습니다.

데이터는 앞 시간과 같은 의료 지식그래프입니다(노드 15,540개 · 관계 91,966개). 지금까지 다룬 20~30개짜리 그래프에서는 인덱스를 만들어도 차이를 느낄 수 없었습니다. 이 규모에서는 다릅니다. 오늘은 그 차이를 **눈으로 재어** 확인합니다.

## ⏪ 복습: 지난 시간까지

- `count`·`sum`·`avg`·`min`·`max` 로 여러 행을 한 줄로 **요약**했습니다.
- RETURN 의 **비집계 항이 그룹핑 키**가 되어 항목별로 묶였습니다.
- `WITH` 로 집계한 값을 넘겨 **다시 집계**하고, `WHERE` 로 **집계 결과를 걸렀습니다.**
- `collect`·`UNWIND` 로 접고 펴고, `SET` 으로 파생 속성을 저장하고, `CASE` 로 라벨을 만들었습니다.
- 오늘은 이 조회들이 **어떤 방법으로 실행되는지**를 들여다보고, 그 방법을 바꾸는 법을 배웁니다.

**오늘의 목표**

**1. 조회를 빠르게: 인덱스**
- [ ] (1-1) `PROFILE`·`EXPLAIN` 으로 **실행계획**을 읽고, 인덱스 전후의 비용을 **직접 잰다**.
- [ ] (1-2) **레이블을 빼면 인덱스를 못 쓴다**는 것을 실행계획으로 확인한다.
- [ ] (1-3) 같은 규칙이 **적재 쿼리에도** 적용돼 수천 배 더 많은 일을 하게 되는 것을 dbHits 로 실측한다.
- [ ] (1-4) 속성을 **함수로 감싸면** 인덱스가 있어도 못 쓴다는 것을 가려낸다.
- [ ] (1-5) 인덱스에 **종류**(RANGE·TEXT)가 있고, 문자열 조건마다 쓸 수 있는 것이 다르다는 것을 안다.

**2. 중복을 막는 규칙: UNIQUE 제약조건**
- [ ] (2-1) `CREATE CONSTRAINT ... IS UNIQUE` 로 중복을 막고, 제약이 **인덱스를 겸한다**는 것을 확인한다.
- [ ] (2-2) 이름이 아니라 **`id` 를 키로 삼아야 하는 이유**를 데이터로 확인한다.
- [ ] (2-3) 제약이 네 가지라는 것과, **내 환경에서 무엇이 되는지** 직접 걸어 본다.

**3. 추천 랭킹: 무엇으로 줄 세울 것인가**
- [ ] (3-1) **가운데 노드를 세는** 집계 랭킹의 정석으로 추천 쿼리를 만든다.
- [ ] (3-2) 바뀌는 값은 **파라미터**로 넘겨 함수 하나로 감싼다.
- [ ] (3-3) **가운데를 바꾸면** 다른 추천이 나온다는 것을 확인한다.
- [ ] (3-4) 그룹마다 **상위 N 개**를 뽑는다(`LIMIT` 과 무엇이 다른지).
- [ ] (3-5) **원점수와 비율이 다른 답을 준다**는 것과, 비율에 **최소 조건**이 필요한 이유를 안다.

오늘 조회할 그래프의 전체 모습입니다. 앞 시간과 같은 의료 지식그래프이고, 오늘은 이 위에서 **인덱스를 걸고 랭킹을 냅니다**. 3절 추천은 가운데에 **유전자**를 두고 셉니다.

<img src="images/그래프_한눈에_의료.png" width="820">

다섯 종류의 노드는 각각 이런 뜻입니다. 이름(레이블)은 데이터 그대로 영어입니다.

| 레이블 | 한글 이름 | 설명 | 예 |
|---|---|---|---|
| `Compound` | 약물 | 약의 유효 성분 하나 | `Carvedilol` |
| `Disease` | 질병 | 병 하나 | `hypertension`(고혈압) |
| `Gene` | 유전자 | 유전자 하나. 그 유전자가 만드는 **단백질까지 이 이름으로** 부릅니다 | `ADRB1` |
| `Symptom` | 증상 | 환자에게 나타나는 증상 | `Edema`(부종) |
| `PharmacologicClass` | 약효분류 | 작용 방식이 같은 약을 묶은 것 | `Calcium Channel Antagonists`(칼슘채널 차단제) |

열두 종류의 관계는 각각 이런 뜻입니다.

| 관계 | 잇는 것 | 설명 |
|---|---|---|
| `INCLUDES` | 약효분류 → 약물 | 그 약이 이 약효분류에 **속한다**(작용 방식이 같은 약 묶음) |
| `TREATS` | 약물 → 질병 | 그 약을 그 병의 **치료제로 쓴다**(병 자체를 겨냥한다) |
| `PALLIATES` | 약물 → 질병 | 그 약을 그 병의 **증상 완화에만 쓴다**(병 자체는 그대로다) |
| `BINDS` | 약물 → 유전자 | 그 약이 그 유전자가 만드는 **단백질에 결합한다**(그 약의 표적) |
| `UPREGULATES_CG` | 약물 → 유전자 | 그 약을 쓰면 그 유전자의 **발현량이 늘어난다** |
| `DOWNREGULATES_CG` | 약물 → 유전자 | 그 약을 쓰면 그 유전자의 **발현량이 줄어든다** |
| `RESEMBLES_CC` | 약물 → 약물 | 두 약의 **화학 구조가 비슷하다** |
| `PRESENTS` | 질병 → 증상 | 그 병의 환자에게 그 **증상이 나타난다** |
| `ASSOCIATES` | 질병 → 유전자 | 그 유전자가 그 병과 **관련 있다고 유전 연구에서 보고됐다** |
| `UPREGULATES_DG` | 질병 → 유전자 | 그 병의 환자에게서 그 유전자의 **발현량이 정상보다 많다** |
| `DOWNREGULATES_DG` | 질병 → 유전자 | 그 병의 환자에게서 그 유전자의 **발현량이 정상보다 적다** |
| `RESEMBLES_DD` | 질병 → 질병 | 두 병이 **닮은 병으로 묶여 있다**(같은 기관이거나 함께 나타나는 일이 잦다) |

> 이름 끝의 `_CG`·`_DG` 는 **무엇과 무엇을 잇는지**를 적어 둔 것입니다. `C` 는 약물(Compound), `D` 는 질병(Disease), `G` 는 유전자(Gene) 입니다. 발현량이 달라진다는 같은 말이라도 **그렇게 만든 것이 약이냐, 그런 상태로 관찰된 것이 병이냐**가 다르므로 이름을 갈라 두었습니다.

**데이터 출처**

- 원본: **Hetionet v1.0** (Himmelstein et al., 2017) · https://het.io · 라이선스 **CC0 1.0**
- 2016년까지 쌓인 생물의학 지식을 한 그래프로 모은 것입니다.
- 여기서 쓰는 것은 원본(노드 47,031 · 관계 2,250,197)에서 관계 24종 가운데 **12종만** 남긴 조각(노드 15,540 · 관계 91,966)입니다.

> **부작용 관계**(어떤 약이 어떤 부작용을 일으키는가)는 **라이선스 때문에** 뺐습니다. 출처인 SIDER 4 가 `CC BY-NC-SA` 라 상업적 이용을 막습니다.

> 뜻풀이는 관계 이름을 우리말로 읽은 것입니다. 관계가 있다는 것은 "그렇다고 **보고된 적이 있다**"는 뜻이지 "그렇다고 입증됐다"는 뜻이 아닙니다.

아래 세 준비 셀을 위에서부터 실행하세요. **연결 → 초기화 → 데이터 적재** 순서입니다. 반드시 **실습 전용 DB**에 연결하세요(초기화 셀이 그래프를 지웁니다).

In [ ]:
# [제공 코드] Neo4j 연결: 실행만 하세요. .env 로 연결하고 run_cypher 헬퍼를 만듭니다.
# 반드시 "실습 전용" DB 에 연결하세요. 아래 실습이 그래프를 지우고 새로 만듭니다.
import os

from dotenv import load_dotenv
from neo4j import GraphDatabase
from neo4j.exceptions import ConstraintError  # UNIQUE 제약 위반 에러

# 1) 접속 정보: .env 를 환경변수로 올린다(비밀번호를 코드에 적지 않으려고)
load_dotenv(".env")       # 같은 폴더의 .env
load_dotenv("../.env")    # 정답 폴더에서 실행하는 경우

# .env 를 못 읽어도 에러가 아니라 기본값으로 넘어간다. 마지막 줄의 주소를 눈으로 확인할 것
NEO4J_URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
NEO4J_USER = os.getenv("NEO4J_USER", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "neo4j")

# 2) 드라이버: 노트북이 끝날 때까지 재사용할 통로 하나
driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))
driver.verify_connectivity()   # 여기서 에러가 나면 주소나 계정이 틀린 것


# 3) 공용 헬퍼: 앞으로 모든 Cypher 는 이 함수 하나로 실행한다
def run_cypher(query, **params):
    """Cypher 실행 -> 결과를 dict 리스트로 반환(수업 공용 헬퍼)."""
    with driver.session() as session:
        # record.data() 가 한 행을 dict 로 바꾼다. 키는 RETURN 의 별칭이다
        return [record.data() for record in session.run(query, **params)]


print("Neo4j 연결:", NEO4J_URI)

In [ ]:
# [제공 코드] 그래프 초기화: 실습 전용 DB 인지 꼭 확인하고 실행하세요!
# 노드·관계에 더해 이 노트북이 만든 인덱스·제약까지 지웁니다(DB 기본 LOOKUP 인덱스는 그대로).
# 1) 제약 먼저. 제약이 남아 있으면 그 제약이 만든 인덱스를 따로 못 지운다
for _row in run_cypher("SHOW CONSTRAINTS YIELD name RETURN name"):
    run_cypher("DROP CONSTRAINT " + _row["name"] + " IF EXISTS")
# 2) 남은 인덱스. LOOKUP 은 DB 기본이라 뺀다
for _row in run_cypher("SHOW INDEXES YIELD name, type WHERE type <> 'LOOKUP' RETURN name"):
    run_cypher("DROP INDEX " + _row["name"] + " IF EXISTS")
# 3) 노드·관계. 관계가 9만 개라 한 번에 담지 않고 2만 개씩 끊어 지운다
while True:
    # DETACH DELETE 는 매달린 관계까지 함께 지운다
    _left = run_cypher("MATCH (n) WITH n LIMIT 20000 DETACH DELETE n RETURN count(n) AS n")[0]["n"]
    if _left == 0:
        break
print("초기화 완료: 남은 노드:", run_cypher("MATCH (n) RETURN count(n) AS n")[0]["n"])

In [ ]:
# [제공 코드] 의료 지식그래프 적재: 실행만 하세요(3초쯤 걸립니다).
# data/ 의 CSV 두 개로 노드 15,540개·관계 91,966개를 만듭니다.
# 맨 앞 CREATE CONSTRAINT 와 MATCH 의 레이블이 왜 필요한지는 1절에서 실측으로 확인합니다.
from pathlib import Path

import pandas as pd

# 교안 폴더에서 실행하면 data/, 정답 폴더에서 실행하면 ../data 를 본다
DATA_DIR = Path("data") if Path("data").exists() else Path("../data")

# 노드 종류(레이블) 5가지. 관계 타입 12가지는 양 끝의 레이블이 하나로 정해져 있다.
NODE_LABELS = ["Compound", "Disease", "Gene", "Symptom", "PharmacologicClass"]
REL_SPEC = {
    "TREATS": ("Compound", "Disease"),              # 약물이 질병을 치료한다(질병 자체를 조절)
    "PALLIATES": ("Compound", "Disease"),           # 약물이 증상을 완화한다(치료와는 다른 말)
    "BINDS": ("Compound", "Gene"),                  # 약물이 그 유전자 산물에 결합한다(표적)
    "UPREGULATES_CG": ("Compound", "Gene"),         # 약물이 유전자 발현을 올린다
    "DOWNREGULATES_CG": ("Compound", "Gene"),       # 약물이 유전자 발현을 내린다
    "RESEMBLES_CC": ("Compound", "Compound"),       # 두 약물의 화학 구조가 닮았다
    "ASSOCIATES": ("Disease", "Gene"),              # 질병과 유전자가 연관돼 있다고 보고됐다
    "UPREGULATES_DG": ("Disease", "Gene"),
    "DOWNREGULATES_DG": ("Disease", "Gene"),
    "RESEMBLES_DD": ("Disease", "Disease"),
    "PRESENTS": ("Disease", "Symptom"),             # 질병이 그 증상으로 나타난다
    "INCLUDES": ("PharmacologicClass", "Compound"),  # 약효분류가 그 약물을 포함한다
}

# 1) id 를 유일하게 만드는 제약을 레이블마다 먼저 건다(2절에서 자세히 배웁니다).
for _label in NODE_LABELS:
    run_cypher(f"CREATE CONSTRAINT {_label.lower()}_id IF NOT EXISTS "
               f"FOR (n:{_label}) REQUIRE n.id IS UNIQUE")
run_cypher("CALL db.awaitIndexes()")   # 제약이 다 준비될 때까지 기다린다

# 2) 노드: 레이블별로 모아 UNWIND 로 한 번에 보낸다.
node_rows = {label: [] for label in NODE_LABELS}   # 레이블 -> 그 레이블 노드 줄 목록
nodes = pd.read_csv(DATA_DIR / "hetionet_nodes.csv")   # 열은 id·name·label 셋
for row in nodes.to_dict("records"):
    node_rows[row["label"]].append({"id": row["id"], "name": row["name"]})
for label, rows in node_rows.items():
    for start in range(0, len(rows), 5000):     # 5,000줄씩 끊어 보낸다(한 번에 다 보내면 메모리를 크게 잡는다)
        # MERGE 는 없으면 만들고 있으면 그대로 둔다. 그래서 이 셀을 두 번 돌려도 노드가 늘지 않는다
        run_cypher(f"UNWIND $rows AS row MERGE (n:{label} {{id: row.id}}) SET n.name = row.name",
                   rows=rows[start:start + 5000])

# 3) 관계: 타입마다 양 끝 레이블을 REL_SPEC 에서 꺼내 MATCH 에 그대로 적는다.
edge_rows = {rel: [] for rel in REL_SPEC}   # 관계 타입 -> 그 타입의 (출발 id, 도착 id) 목록
edges = pd.read_csv(DATA_DIR / "hetionet_edges.csv")   # 열은 source·rel·target 셋
for row in edges.to_dict("records"):
    edge_rows[row["rel"]].append({"s": row["source"], "t": row["target"]})
for rel, rows in edge_rows.items():
    source_label, target_label = REL_SPEC[rel]   # 이 레이블을 MATCH 에 적어야 노드를 곧장 짚는다
    for start in range(0, len(rows), 5000):
        run_cypher(f"UNWIND $rows AS row "
                   f"MATCH (a:{source_label} {{id: row.s}}), (b:{target_label} {{id: row.t}}) "
                   f"MERGE (a)-[:{rel}]->(b)", rows=rows[start:start + 5000])

print("적재 완료: 노드", run_cypher("MATCH (n) RETURN count(n) AS n")[0]["n"],
      "· 관계", run_cypher("MATCH ()-[r]->() RETURN count(r) AS n")[0]["n"])

---
# 1. 조회를 빠르게: 인덱스

인덱스는 다섯 걸음으로 익힙니다. 먼저 인덱스를 **만들어 실행계획으로 재고**(1-1), 그다음 인덱스를 **못 쓰게 되는 경우** 셋을 차례로 봅니다. 레이블을 빼면 못 쓰고(1-2), 그 규칙은 **적재 쿼리에도 그대로** 적용되며(1-3), 속성을 **함수로 감싸도** 못 씁니다(1-4). 마지막으로 인덱스에도 **종류가 있다**는 것을 봅니다(1-5).

## 1-1. 인덱스를 만들고 실행계획으로 재기

### 왜 필요할까요?
`MATCH (g:Gene {name: 'ALB'})` 이라고 쓰면 데이터베이스는 무엇을 할까요? 이 그래프의 `Gene` 노드는 **13,113개**입니다. 아무 준비가 없으면 그 전부를 하나씩 꺼내 이름을 맞춰 봐야 합니다.

**인덱스**는 미리 만들어 둔 찾아보기입니다. 책 뒤의 찾아보기처럼, 이름에서 노드 위치로 바로 건너뛰게 해 줍니다.

### 문법: 인덱스 만들기·확인
| 표현 | 하는 일 |
|---|---|
| `CREATE INDEX 이름 IF NOT EXISTS FOR (n:레이블) ON (n.속성)` | 인덱스를 만든다 |
| `CALL db.awaitIndexes()` | 인덱스가 다 만들어질 때까지 기다린다 |
| `SHOW INDEXES` | 지금 있는 인덱스 목록 |
| `DROP INDEX 이름 IF EXISTS` | 인덱스를 지운다 |

### 언제 만드나요?
- **자주 조회 조건으로 쓰는 속성**에 만듭니다(`name`, `id` 등).
- 공짜가 아닙니다. 데이터를 넣고 고칠 때마다 인덱스도 함께 갱신되고, 저장 공간도 씁니다.
- 그래서 **모든 속성에 다 거는 것이 아니라, 실제로 찾는 데 쓰는 속성에만** 겁니다.

### 정말 빨라지는지 어떻게 확인하나요?

"몇 초 걸렸다"로 재면 실행할 때마다 값이 흔들려 믿을 수 없습니다. 대신 **실행계획**을 봅니다. 실행계획은 데이터베이스가 그 쿼리를 **어떤 방법으로** 풀었는지를 보여 줍니다.

| 붙이는 말 | 하는 일 |
|---|---|
| `EXPLAIN 쿼리` | 실행하지 **않고** 계획만 본다 |
| `PROFILE 쿼리` | 실제로 실행하면서 **비용까지** 잰다 |

비용은 **dbHits**(저장소를 들여다본 횟수)로 나옵니다. 이 수는 실행할 때마다 같으므로 **되풀이해 재도 같은 답**이 나옵니다. 아래 헬퍼가 계획을 보기 좋게 찍어 줍니다.

In [ ]:
# [제공 코드] 실행계획 보기 헬퍼: 실행만 하세요.
# explain_plan(쿼리, profile=False) 는 계획만, True 는 비용(dbHits)까지. quiet=True 면 찍지 않고 값만.
PLAN_CALLS = []   # 부른 기록: (쿼리, profile 여부, 연산자 목록).
#                 과제 채점이 "정말 재 봤는지"와 "그 결과를 담았는지"를 볼 때 쓴다


def explain_plan(query, profile=True, quiet=False, **params):
    """실행계획을 출력하고 (연산자 이름 리스트, 총 dbHits) 를 돌려준다."""
    # profile=True 는 PROFILE(실제로 실행하며 dbHits 측정), False 는 EXPLAIN(실행 없이 계획만)
    with driver.session() as session:
        if profile:
            result = session.run("PROFILE " + query, **params)
            list(result)                      # PROFILE 은 결과를 끝까지 읽어야 계측이 끝난다
            plan = result.consume().profile   # 계획은 실행이 끝난 뒤에야 받을 수 있다
        else:
            plan = session.run("EXPLAIN " + query, **params).consume().plan
    total = 0        # 계획 전체의 dbHits 합
    operators = []   # 위에서부터 만난 연산자 이름들. 뒤에서 'NodeIndexSeek 이 있나' 를 볼 때 쓴다

    # 실행계획은 나무 모양이라, 자기 자신을 다시 부르며 아래로 내려간다
    def walk(node, depth=0):
        nonlocal total
        hits = node.get("dbHits")                   # EXPLAIN 으로 뽑은 계획에는 이 값이 없다(None)
        total += hits or 0
        name = node["operatorType"].split("@")[0]   # 'NodeIndexSeek@neo4j' 에서 이름만 남긴다
        operators.append(name)
        cost = f"| dbHits = {hits}" if profile else ""
        if not quiet:                     # quiet=True 면 재기만 하고 찍지 않는다
            print("  " * depth, name, cost)
        for child in node.get("children", []):
            walk(child, depth + 1)        # 자식 계획은 한 칸 더 들여써 찍는다

    walk(plan)
    # 무엇을 쟀고 무엇이 나왔는지 남긴다(채점이 손으로 적은 목록을 걸러 낼 때 본다)
    PLAN_CALLS.append((query, profile, operators))
    if profile and not quiet:
        print("총 dbHits:", total)
    return operators, total

In [ ]:
# 인덱스를 만들기 '전': 이름으로 유전자 하나를 찾는 조회가 어떤 방법으로 실행되는지 본다
# 돌려받는 두 값(연산자 목록·총 dbHits)은 인덱스를 만든 뒤와 견주는 데 쓴다
ops_before, hits_before = explain_plan("MATCH (g:Gene {name: 'ALB'}) RETURN g.id AS id")

- **`NodeByLabelScan`**: `Gene` 레이블이 붙은 노드를 **전부 훑었다**는 뜻입니다.
- **총 dbHits 26,228**: 유전자 13,113개를 꺼내고, 그 이름을 하나하나 맞춰 본 값입니다.

결과는 한 줄뿐인데 일한 양은 이만큼입니다. 이제 찾아보기를 만들어 주겠습니다.

In [ ]:
# IF NOT EXISTS 를 붙이면 이미 있어도 에러가 나지 않는다(여러 번 실행해도 안전)
run_cypher("CREATE INDEX gene_name IF NOT EXISTS FOR (g:Gene) ON (g.name)")
# 인덱스 만들기는 뒤에서 따로 돌아간다. 기다리지 않고 바로 재면 아직 안 쓰이는 상태가 잡힌다
run_cypher("CALL db.awaitIndexes()")
# SHOW INDEXES 는 있는 것을 전부 내놓는다. name 으로 걸러 방금 만든 것만 본다
for row in run_cypher("""
    SHOW INDEXES YIELD name, type, labelsOrTypes, properties
    WHERE name = 'gene_name' RETURN name, type, labelsOrTypes, properties
"""):
    print(row)

In [ ]:
# 인덱스를 만든 '후': 똑같은 조회를 다시 본다 (쿼리는 한 글자도 바꾸지 않았다)
ops_after, hits_after = explain_plan("MATCH (g:Gene {name: 'ALB'}) RETURN g.id AS id")
print()
print("인덱스 전:", hits_before, "-> 인덱스 후:", hits_after,
      "=", round(hits_before / hits_after), "배")

<img src="images/인덱스_스캔_대_시크.png" width="760">

쿼리도 결과도 그대로인데 **일한 양만** `26,228` 에서 `3` 으로 줄었습니다. 인덱스는 답을 바꾸는 것이 아니라 **찾는 방법**을 바꿉니다.

연산자 이름이 `NodeByLabelScan` 에서 **`NodeIndexSeek`** 으로 바뀌었습니다. "레이블 전체를 훑는다"에서 "찾아보기로 바로 짚는다"로 방법이 달라진 것입니다. 약 **8,743배**입니다.

> 앞 단원까지 쓰던 20~30개짜리 연습 그래프였다면 이 차이가 `20` 대 `3` 쯤이라 "그래서 뭐" 소리가 나왔을 것입니다. 인덱스는 **데이터가 클수록** 값어치가 커집니다.

### 계획만 보고 싶을 때: `EXPLAIN`

`PROFILE` 은 쿼리를 실제로 실행합니다. 오래 걸리거나 데이터를 바꾸는 쿼리라면 곤란합니다. 그럴 때는 `EXPLAIN` 으로 **실행 없이 계획만** 봅니다(대신 dbHits 는 나오지 않습니다).

In [ ]:
# EXPLAIN: 실행하지 않고 '계획만' 본다 (PROFILE 과 달리 dbHits 가 없다)
# 데이터를 바꾸는 쿼리(CREATE·DELETE)에 안심하고 붙일 수 있는 쪽이 이것이다
explain_plan("MATCH (g:Gene {name: 'ALB'}) RETURN g.id AS id", profile=False)

---
## 1-2. 레이블을 빼면 인덱스를 못 쓴다

여기가 오늘 가장 중요한 대목입니다.

방금 만든 인덱스는 `FOR (g:Gene) ON (g.name)` 이었습니다. **`Gene` 이라는 레이블에 걸린** 인덱스입니다. 그러니 레이블을 적지 않은 조회는 이 인덱스를 쓸 수 없습니다. 찾는 값이 똑같아도 그렇습니다.

In [ ]:
# 앞의 조회에서 레이블(:Gene)만 뺐다. 찾는 이름은 그대로다
ops_nolabel, hits_nolabel = explain_plan("MATCH (n {name: 'ALB'}) RETURN n.id AS id")
print()
print("레이블 적음:", hits_after, "| 레이블 뺌:", hits_nolabel)

<img src="images/레이블별_인덱스.png" width="760">

인덱스는 **레이블 하나에 매달려** 있습니다. 조회에 그 레이블이 없으면 데이터베이스는 어느 인덱스를 써야 할지 알 수 없어, 그래프 전체를 훑습니다.

- 레이블을 적었을 때: **`NodeIndexSeek` / dbHits 3**
- 레이블을 뺐을 때: **`AllNodesScan` / dbHits 31,082**

`AllNodesScan` 은 "그래프의 **모든 노드**를 훑었다"는 뜻입니다. 인덱스가 있는데도 인덱스 없을 때(26,228)보다 **더 많이** 일했습니다. 훑을 범위가 `Gene` 13,113개에서 전체 15,540개로 늘었기 때문입니다.

**레이블은 장식이 아니라 검색 범위입니다.** 패턴을 쓸 때 레이블을 습관처럼 적으세요.

---
## 1-3. 이 규칙은 적재 쿼리에도 그대로 적용된다

이 노트북 맨 위에서 그냥 실행하고 넘어간 적재 셀을 다시 보겠습니다. 관계를 만드는 부분이 이렇게 생겼습니다.

```
UNWIND $rows AS row
MATCH (a:Compound {id: row.s}), (b:Disease {id: row.t})
MERGE (a)-[:TREATS]->(b)
```

그리고 맨 앞에서 레이블마다 이런 제약을 걸어 두었습니다(2절에서 자세히 봅니다).

```
CREATE CONSTRAINT compound_id IF NOT EXISTS FOR (n:Compound) REQUIRE n.id IS UNIQUE
```

**이 제약이 `Compound.id` 인덱스를 함께 만들어 줍니다.** 그래서 위 `MATCH` 가 15,540개를 훑지 않고 곧바로 노드를 짚습니다.

레이블을 빼면 어떻게 될까요. `TREATS` 관계 200줄로 재어 보겠습니다. `MERGE` 는 이미 있으면 그대로 두므로 여러 번 실행해도 관계가 늘지 않습니다.

In [ ]:
# [제공 코드] 적재 비교용 데이터: CSV 에서 TREATS 관계 앞 200줄만 읽어 옵니다(실행만 하세요).
from pathlib import Path

import pandas as pd

_DATA_DIR = Path("data") if Path("data").exists() else Path("../data")
# rel 이 TREATS 인 줄만 골라 앞에서 200줄. 200 은 차이가 뚜렷하면서 느린 쪽도 1초 안에 끝나는 크기다
_edges = pd.read_csv(_DATA_DIR / "hetionet_edges.csv", dtype=str)
_treats = _edges[_edges["rel"] == "TREATS"].head(200)
sample_rows = [{"s": r["source"], "t": r["target"]}
               for r in _treats.to_dict("records")]
print("비교에 쓸 줄 수:", len(sample_rows))
print("첫 줄:", sample_rows[0])

In [ ]:
# 적재 셀이 실제로 쓴 모양: 양 끝에 레이블을 적었다
# 레이블이 있으니 제약이 만들어 준 id 인덱스를 타고 줄마다 노드를 곧장 짚는다
ops_load_ok, hits_load_ok = explain_plan("""
    UNWIND $rows AS row
    MATCH (a:Compound {id: row.s}), (b:Disease {id: row.t})
    MERGE (a)-[:TREATS]->(b)
""", rows=sample_rows)

In [ ]:
# 레이블만 뺀 같은 적재. 이 셀은 1초쯤 걸립니다(느려진 것이 이 절의 요점입니다)
# 레이블이 없어 인덱스를 못 쓰고, 줄마다 노드 15,540개를 양 끝으로 두 번씩 훑는다
ops_load_bad, hits_load_bad = explain_plan("""
    UNWIND $rows AS row
    MATCH (a {id: row.s}), (b {id: row.t})
    MERGE (a)-[:TREATS]->(b)
""", rows=sample_rows)
print()
print("레이블 적음:", f"{hits_load_ok:,}", "| 레이블 뺌:", f"{hits_load_bad:,}",
      "->", f"{round(hits_load_bad / hits_load_ok):,}", "배")

<img src="images/적재_레이블_비용.png" width="760">

200줄을 넣는 데 드는 일의 양입니다. 레이블을 적으면 인덱스로 바로 짚고, 빼면 줄마다 그래프 전체를 두 번씩(양 끝 노드) 훑습니다. 막대 길이는 차이를 보이기 위한 것이라 실제 비례가 아닙니다.

200줄에 **3,034배** 차이입니다. 앞의 조회 하나와 똑같은 이유입니다. 레이블이 없으니 인덱스를 못 쓰고, 줄마다 노드 15,540개를 양 끝으로 두 번씩 훑습니다.

이 셀은 200줄만 넣었습니다. 실제 적재는 **91,966줄**이고, 그때는 이 차이가 몇 초와 몇 분의 차이로 나타납니다. 그런데도 **에러는 나지 않습니다.** 그냥 느릴 뿐입니다. 이런 종류의 실수가 제일 찾기 어렵습니다.

> 전체 적재를 두 방식으로 돌려 **시간을 직접 재는 것**은 다음 시간(지식그래프 적재)에서 합니다. 오늘은 "왜 그렇게 되는가"까지만 확인합니다.

**조회든 적재든 느리면 제일 먼저 볼 것**: 조건에 쓴 속성에 인덱스가 있는가, 그리고 그 패턴에 **레이블을 적었는가**.

### 🖐️ 함께 따라하기: 약물 이름 인덱스를 전후로 대조하기

데모는 `Gene` 레이블로 했습니다. 따라하기는 `Compound`(약물 1,531개)로 같은 일을 합니다.

1. `MATCH (c:Compound {name: 'Warfarin'}) RETURN c.id AS id` 를 `explain_plan` 으로 재고 연산자·dbHits 를 확인합니다(인덱스 없는 상태).
2. `CREATE INDEX compound_name IF NOT EXISTS FOR (c:Compound) ON (c.name)` 로 인덱스를 만들고 `CALL db.awaitIndexes()` 로 기다립니다.
3. 같은 조회를 다시 재고, 두 dbHits 를 나란히 출력합니다.

확인 기준: 인덱스 전 dbHits 는 `3,064`(`NodeByLabelScan`), 후에는 `3`(`NodeIndexSeek`)입니다. `Gene` 일 때(26,228)보다 전(前) 값이 작은 이유도 생각해 보세요.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) 인덱스 없는 상태로 explain_plan 을 돌려 결과를 ops1, hits1 두 변수로 받는다
# 2) CREATE INDEX compound_name IF NOT EXISTS FOR (c:Compound) ON (c.name) 실행
# 3) CALL db.awaitIndexes() 로 기다린다
# 4) 같은 조회를 다시 재어 ops2, hits2 에 담고, 전/후 dbHits 와 몇 배인지 함께 print

> `Gene` 은 13,113개, `Compound` 는 1,531개입니다. 훑어야 할 노드가 적으니 인덱스 없을 때의 비용도 그만큼 작습니다. **인덱스의 값어치는 그 레이블의 노드가 많을수록 커집니다.**

### ✅ 바로 확인 퀴즈

**1)** `Gene.name` 에 인덱스를 만들었습니다. `MATCH (c:Compound {name: 'Warfarin'})` 은 이 인덱스를 쓸까요?

<details><summary>정답 보기</summary>

쓰지 못합니다. 인덱스는 **레이블 하나에 매달려** 있습니다. `Compound` 로 찾으려면 `Compound.name` 인덱스를 따로 만들어야 합니다.

</details>

**2)** `PROFILE` 과 `EXPLAIN` 중 데이터를 바꾸는 쿼리에 안심하고 붙일 수 있는 것은?

<details><summary>정답 보기</summary>

`EXPLAIN`. 실행하지 않고 계획만 봅니다. `PROFILE` 은 실제로 실행하므로 `CREATE`·`DELETE` 같은 쿼리에 붙이면 데이터가 정말로 바뀝니다.

</details>

**3)** 인덱스를 모든 속성에 다 걸면 안 되는 이유는?

<details><summary>정답 보기</summary>

인덱스는 데이터를 넣고 고칠 때마다 함께 갱신돼야 하고 저장 공간도 씁니다. 찾는 데 쓰지 않는 속성의 인덱스는 **쓰기만 느리게 하고 얻는 것이 없습니다.**

</details>

**4)** 동료가 "적재 스크립트가 한참을 돌아도 안 끝난다"고 합니다. 에러는 없습니다. 무엇부터 확인하라고 하겠습니까?

<details><summary>정답 보기</summary>

**적재 쿼리의 `MATCH` 에 레이블이 적혀 있는지**, 그리고 **찾는 속성에 인덱스나 제약이 있는지**. 둘 중 하나만 빠져도 줄마다 그래프 전체를 훑습니다. 에러가 안 나는 것이 이 실수의 특징이라, "어딘가 잘못됐다"는 신호가 오지 않습니다. 먼저 몇백 줄만 잘라 `PROFILE` 로 dbHits 를 재 보면 바로 드러납니다.

</details>

---
## 1-4. 속성을 함수로 감싸도 인덱스를 못 쓴다

### 왜 필요할까요?
레이블은 잘 적었습니다. 그런데도 느린 경우가 하나 더 있습니다. 실무에서 아주 흔한 모양입니다.

```
MATCH (g:Gene) WHERE toLower(g.name) = 'alb'      대소문자를 무시하고 찾고 싶다
```

찾는 값도 같고 레이블도 적었는데, 이 조회는 1-1 에서 만든 `gene_name` 인덱스를 **쓰지 못합니다.** 재어 보겠습니다.

In [ ]:
# 속성을 toLower() 로 감싼 같은 조회. 레이블도 적었고 인덱스도 그대로 있다
# 인덱스는 '저장된 값' 에 걸린다. toLower(g.name) 은 읽어 온 뒤에야 계산되는 값이라 인덱스에 없다
ops_wrap, hits_wrap = explain_plan("MATCH (g:Gene) WHERE toLower(g.name) = 'alb' RETURN g.id AS id")
print()
print("인덱스 없을 때(1-1):", hits_before,
      "| 인덱스가 있어도 감싸면:", hits_wrap,
      "| 그냥 찾기:", hits_after)

함수로 감싼 조회의 dbHits 는 `26,228` 입니다. 1-1 에서 **인덱스가 없을 때** 잰 `26,228` 과 같은 값입니다. 인덱스를 만든 것이 이 조회에는 아무 도움이 되지 않았습니다.

인덱스는 **저장된 값**을 늘어놓은 찾아보기입니다. `toLower(g.name)` 은 노드를 꺼내 온 뒤에야 계산되는 값이라 찾아보기에 실려 있지 않습니다. 그래서 데이터베이스는 레이블 전체를 훑어 한 줄씩 계산해 볼 수밖에 없습니다.

**대소문자를 무시하고 찾아야 한다면**, 소문자로 바꾼 값을 속성으로 **한 벌 저장해 두고**(`SET g.name_lower = toLower(g.name)`) 그 속성에 인덱스를 겁니다. 앞 시간의 **파생 속성**이 바로 이 자리에서 쓰입니다.

> 같은 이유로 `WHERE g.name + '!' = 'ALB!'` 나 `WHERE substring(g.name, 0, 3) = 'ALB'` 도 인덱스를 못 씁니다. **비교식의 왼쪽에 속성이 맨몸으로 있어야** 합니다.

<img src="images/함수로_감싸면_인덱스를_못_쓴다.png" width="760">

인덱스에 실려 있는 것은 **저장된 값 그대로**입니다. 함수를 씌운 값은 노드를 꺼내 온 뒤에야 생기므로 찾아보기에 없습니다.

### ✅ 바로 확인 퀴즈

**1)** `MATCH (g:Gene) WHERE toUpper(g.name) = 'ALB'` 는 `Gene.name` 인덱스를 쓸까요?

<details><summary>정답 보기</summary>

쓰지 못합니다. 속성을 함수로 감싼 순간 **저장된 값과 비교하는 것이 아니게** 됩니다. 연산자는 `NodeByLabelScan` 으로 떨어집니다.

</details>

**2)** 그런데도 대소문자를 무시하고 빠르게 찾아야 한다면 어떻게 하나요?

<details><summary>정답 보기</summary>

소문자로 바꾼 값을 **파생 속성으로 저장**하고 그 속성에 인덱스를 겁니다. 찾을 때는 `WHERE g.name_lower = toLower($찾을이름)` 처럼 **속성 쪽은 맨몸으로** 두고 파라미터 쪽만 소문자로 바꿉니다.

</details>

---
## 1-5. 인덱스에도 종류가 있다: RANGE 와 TEXT

### 왜 필요할까요?
지난 단원(Cypher 심화)에서 문자열을 네 가지 방법으로 걸렀습니다. 같다(`=`), 앞이 같다(`STARTS WITH`), 뒤가 같다(`ENDS WITH`), 어딘가에 있다(`CONTAINS`). 지금 만든 인덱스가 이 넷을 다 빠르게 해 줄까요?

`ALB` 조각으로 네 가지를 모두 물어보겠습니다. 걸리는 유전자 수가 서로 다릅니다(같음 1개 · 앞맞춤 1개 · 뒷맞춤 3개 · 부분일치 8개). **다른 질문**이라는 뜻입니다.

### 문법: 인덱스 종류
| 표현 | 하는 일 |
|---|---|
| `CREATE INDEX ...` | **RANGE** 인덱스(기본값). 값을 사전 순으로 늘어놓는다 |
| `CREATE TEXT INDEX ...` | **TEXT** 인덱스. 문자열 조각으로 찾는 데 맞춰져 있다 |

이름을 사전 순으로 늘어놓으면 `ALB` 로 **시작**하는 것은 한자리에 모여 있어 바로 짚을 수 있습니다. 하지만 `ALB` 로 **끝나는** 것과 `ALB` 를 **품은** 것은 사전 어디에나 흩어져 있습니다. 그래서 RANGE 인덱스로는 그 둘을 짚지 못합니다.

<img src="images/range_대_text_인덱스.png" width="760">

사전 순으로 늘어놓았기 때문에 **앞맞춤은 한자리에 모이고 부분일치는 흩어집니다.** 인덱스 종류마다 풀 수 있는 조건이 다른 이유가 이 한 가지입니다. 그림에는 흩어진 것 가운데 일부만 그렸습니다(실제로는 더 많습니다).

In [ ]:
# [제공 코드] 문자열 조건 네 가지를 한 번에 재는 헬퍼입니다(실행만 하세요).
# 같은 네 조회를 인덱스 상태만 바꿔 가며 세 번 재고, dbHits 를 나란히 모읍니다.
TEXT_QUERIES = {
    "같음": "MATCH (g:Gene) WHERE g.name = 'ALB' RETURN count(g) AS n",
    "앞맞춤": "MATCH (g:Gene) WHERE g.name STARTS WITH 'ALB' RETURN count(g) AS n",
    "뒷맞춤": "MATCH (g:Gene) WHERE g.name ENDS WITH 'ALB' RETURN count(g) AS n",
    "부분일치": "MATCH (g:Gene) WHERE g.name CONTAINS 'ALB' RETURN count(g) AS n",
}


def sweep_text(label):
    """네 조회의 (연산자, dbHits) 를 재어 dict 로 돌려주고 한 줄씩 찍는다."""
    out = {}
    print(f"[{label}]")
    for key, query in TEXT_QUERIES.items():
        # quiet=True: 계획 나무는 찍지 않고 값만 받는다(네 개를 한 표로 볼 참이라)
        ops, hits = explain_plan(query, quiet=True)
        # ops 의 마지막 원소가 실제로 노드를 찾아 오는 연산자다(계획 나무의 잎)
        out[key] = (ops[-1], hits)
        # 한글은 글자 폭이 달라 자릿수를 못 맞춘다. 자릿수 맞출 값은 앞에, 한글 이름은 뒤에 둔다
        print(f"  {ops[-1]:<24} dbHits = {hits:>9,}   {key}")
    return out

In [ ]:
# 1) 인덱스를 둘 다 지운 상태. 넷 다 레이블 전체를 훑는다
run_cypher("DROP INDEX gene_name IF EXISTS")
run_cypher("DROP INDEX gene_name_text IF EXISTS")
no_index = sweep_text("인덱스 없음")

In [ ]:
# 2) RANGE 인덱스(우리가 1-1 에서 만든 그것)만 있는 상태
run_cypher("CREATE INDEX gene_name IF NOT EXISTS FOR (g:Gene) ON (g.name)")
run_cypher("CALL db.awaitIndexes()")
range_only = sweep_text("RANGE 인덱스")

In [ ]:
# 3) TEXT 인덱스를 더한 상태. 같은 속성에 종류가 다른 인덱스를 함께 둘 수 있다
run_cypher("CREATE TEXT INDEX gene_name_text IF NOT EXISTS FOR (g:Gene) ON (g.name)")
run_cypher("CALL db.awaitIndexes()")
range_text = sweep_text("RANGE + TEXT")

In [ ]:
# 세 번 잰 결과를 한 표로 모아 본다. 아래 마크다운 표와 같은 값이 나와야 한다
for key in TEXT_QUERIES:
    print(f"{key:<6} 인덱스 없음 {no_index[key][1]:>9,}"
          f"   RANGE {range_only[key][1]:>9,} {range_only[key][0]:<26}"
          f"   RANGE+TEXT {range_text[key][1]:>9,} {range_text[key][0]}")

세 번 잰 것을 한 표로 모으면 이렇습니다.

| 조회 | 인덱스 없음 | RANGE | RANGE + TEXT |
|---|---|---|---|
| `같음` | 26,227 | 2 `NodeIndexSeek` | 2 `NodeIndexSeek` |
| `앞맞춤` | 26,227 | 2 `NodeIndexSeekByRange` | 2 `NodeIndexSeekByRange` |
| `뒷맞춤` | 26,227 | 13,114 `NodeIndexScan` | 4 `NodeIndexEndsWithScan` |
| `부분일치` | 26,227 | 13,114 `NodeIndexScan` | 9 `NodeIndexContainsScan` |

**RANGE 인덱스는 뒷맞춤·부분일치를 짚지 못합니다.** 연산자가 `NodeIndexScan` 인 것은 "인덱스에 실린 이름을 처음부터 끝까지 훑었다"는 뜻입니다. 노드를 꺼내지는 않아 값이 절반쯤으로 줄었을 뿐, **전부 훑는 것은 똑같습니다.**

TEXT 인덱스를 더하자 `NodeIndexEndsWithScan`·`NodeIndexContainsScan` 로 바뀌면서 13,114 가 9 가 됐습니다.

> 표의 `같음` 이 1-1 에서 잰 26,228·3 과 하나씩 다른 것은, 여기서는 `count(g)` 만 세고 1-1 에서는 `g.id` 까지 꺼냈기 때문입니다. 꺼내는 값이 하나 늘면 들여다보는 횟수도 하나 늘어납니다.

### 인덱스는 관계 속성에도 걸립니다

지금까지는 노드만 봤습니다. 관계에 붙은 속성도 조회 조건이 될 수 있고, 그럴 때는 관계 인덱스를 만듭니다. 괄호 모양만 노드에서 관계로 바뀝니다.

```
CREATE INDEX 이름 IF NOT EXISTS FOR ()-[r:관계타입]-() ON (r.속성)
```

이 그래프의 관계에는 속성이 없어 실습할 자리가 없지만, 과제에서 쓰는 `ORDERED {qty, price}`·`PLAYED {cnt}` 같은 관계가 바로 그 대상입니다.

In [ ]:
# 관계 속성에도 인덱스를 걸 수 있다는 것만 확인한다(이 그래프에는 그 속성이 없어 쓰이지는 않는다)
run_cypher("CREATE INDEX treats_since IF NOT EXISTS FOR ()-[r:TREATS]-() ON (r.since)")
run_cypher("CALL db.awaitIndexes()")
# entityType 열이 NODE 인지 RELATIONSHIP 인지로 무엇에 걸린 인덱스인지 갈린다
for row in run_cypher("""
    SHOW INDEXES YIELD name, type, entityType, labelsOrTypes, properties
    WHERE name = 'treats_since'
    RETURN name, type, entityType, labelsOrTypes, properties
"""):
    print(row)

In [ ]:
# 확인만 했으니 지운다. 쓰지 않는 인덱스는 쓰기만 느리게 한다(1-1 에서 본 그대로)
run_cypher("DROP INDEX treats_since IF EXISTS")

### 🖐️ 함께 따라하기: 약물 이름을 부분일치로 찾아보기

데모는 `Gene` 로 했습니다. 따라하기는 `Compound`(약물 1,531개)에서 **이름에 `statin` 이 들어간 약**을 찾습니다. 1-3 따라하기에서 이미 `compound_name` RANGE 인덱스를 만들어 두었습니다.

1. `MATCH (c:Compound) WHERE c.name CONTAINS 'statin' RETURN count(c) AS n` 을 `explain_plan` 으로 재고 연산자·dbHits 를 확인합니다.
2. `CREATE TEXT INDEX compound_name_text IF NOT EXISTS FOR (c:Compound) ON (c.name)` 를 만들고 `CALL db.awaitIndexes()` 로 기다립니다.
3. 같은 조회를 다시 재고 두 dbHits 를 나란히 출력합니다.

확인 기준: RANGE 만 있을 때는 연산자가 `NodeIndexScan` 이고, TEXT 를 더하면 `NodeIndexContainsScan` 으로 바뀌며 dbHits 가 크게 줄어듭니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) statin_q 에 부분일치 조회를 담고 explain_plan 으로 재어 ops1, hits1 두 변수로 받는다
# 2) CREATE TEXT INDEX compound_name_text ... FOR (c:Compound) ON (c.name) 실행
# 3) CALL db.awaitIndexes() 로 기다린다
# 4) 같은 조회를 다시 재어 ops2, hits2 에 담고, 전/후 dbHits 를 나란히 print

### ✅ 바로 확인 퀴즈

**1)** `CREATE INDEX` 로 만든 인덱스는 어떤 종류인가요?

<details><summary>정답 보기</summary>

**RANGE** 입니다(기본값). `SHOW INDEXES` 의 `type` 열에서 확인할 수 있습니다. `CREATE TEXT INDEX` 라고 적어야 TEXT 인덱스가 만들어집니다.

</details>

**2)** 이름이 `-statin` 으로 끝나는 약을 자주 찾는다면 어떤 인덱스가 필요한가요?

<details><summary>정답 보기</summary>

**TEXT** 인덱스입니다. `ENDS WITH` 와 `CONTAINS` 는 RANGE 인덱스로 짚을 수 없습니다. 반대로 `STARTS WITH` 와 `=` 는 RANGE 로 충분합니다.

</details>

**3)** 그러면 모든 문자열 속성에 TEXT 인덱스를 하나씩 더 걸어 두면 되지 않나요?

<details><summary>정답 보기</summary>

아닙니다. 인덱스는 하나 늘 때마다 **쓰기가 느려지고 저장 공간을 씁니다**(1-1 에서 본 그대로). 게다가 TEXT 인덱스는 크기 비교(`<`, `>=`)를 풀지 못합니다(앞맞춤은 RANGE 와 똑같이 짚습니다). **그 속성을 실제로 부분일치로 찾고 있을 때만** 더합니다.

</details>

---
# 2. 중복을 막는 규칙: UNIQUE 제약조건

제약조건은 세 걸음입니다. 제약을 **걸고 무엇이 딸려 오는지 확인하고**(2-1), 왜 이름이 아니라 **`id` 를 키로 삼는지** 데이터로 보고(2-2), 제약에 **네 가지 종류**가 있다는 것과 내 환경에서 무엇이 되는지 직접 걸어 봅니다(2-3).

## 2-1. 제약을 걸고 확인하기

### 왜 필요할까요?
인덱스는 **빨리 찾게** 해 주지만 **중복을 막지는 못합니다.** 같은 약물이 두 번 들어오면 인덱스는 둘 다 성실히 색인합니다. 집계는 그때부터 조용히 틀린 답을 냅니다.

**제약조건**은 규칙을 어기는 데이터를 아예 **거부**합니다.

### 문법: UNIQUE 제약
| 표현 | 하는 일 |
|---|---|
| `CREATE CONSTRAINT 이름 IF NOT EXISTS FOR (n:레이블) REQUIRE n.속성 IS UNIQUE` | 유일성 제약 |
| `SHOW CONSTRAINTS` | 지금 있는 제약 목록 |
| `DROP CONSTRAINT 이름 IF EXISTS` | 제약을 지운다 |

<img src="images/인덱스_대_제약.png" width="760">

인덱스는 찾아보기, 제약은 문지기입니다. 그리고 **제약을 걸면 인덱스는 덤으로 따라옵니다.**

In [ ]:
# 적재 셀이 이미 만들어 둔 제약들. 레이블마다 id 를 유일하게 하는 규칙이다
# ORDER BY name 으로 차례를 고정해 실행할 때마다 같은 순서로 보이게 한다
for row in run_cypher("""
    SHOW CONSTRAINTS YIELD name, type, labelsOrTypes, properties
    RETURN name, type, labelsOrTypes, properties ORDER BY name
"""):
    print(row)

In [ ]:
# 제약은 인덱스를 겸한다: 제약이 만들어 준 인덱스만 골라 본다
# owningConstraint 에 그 인덱스를 만든 제약의 이름이 들어 있다. 비어 있으면 손으로 만든 인덱스다
for row in run_cypher("""
    SHOW INDEXES YIELD name, type, owningConstraint
    WHERE owningConstraint IS NOT NULL
    RETURN name, type, owningConstraint ORDER BY name
"""):
    print(row)

제약 5개마다 같은 이름의 `RANGE` 인덱스가 딸려 있습니다. 이것이 1절에서 적재 `MATCH` 가 빨랐던 이유입니다. **`id` 에 인덱스를 따로 만든 적이 없는데도** 제약이 만들어 준 인덱스가 일하고 있었던 것입니다.

> 그래서 실무에서는 "유일해야 하는 키"에는 인덱스가 아니라 **제약**을 겁니다. 중복도 막고 인덱스도 얻으니 한 번에 두 가지가 해결됩니다.

### 규칙이 실제로 막는지 확인하기

`Warfarin` 는 이미 `Compound::DB00682` 라는 `id` 로 들어 있습니다. 같은 `id` 로 노드를 하나 더 만들어 보겠습니다. 제약이 일한다면 **에러가 나야** 합니다.

In [ ]:
# 제약이 있으면 이 시도는 ConstraintError 로 거부된다
try:
    run_cypher("CREATE (:Compound {id: 'Compound::DB00682', name: '중복 시도'})")
    print("만들어졌습니다. 제약이 걸려 있지 않다는 뜻입니다")
except ConstraintError as error:
    # Exception 으로 넓게 잡으면 오타·연결 끊김 같은 다른 에러까지 '성공'으로 삼킨다
    print("제약이 막았습니다")
    # 에러 메시지가 길어 읽기 힘들다. '}' 뒤부터 앞 90자만 잘라 보여 준다
    print("  ", str(error).split("}")[1].strip()[:90])

In [ ]:
# 노드 수가 늘지 않았는지 확인한다
print(run_cypher("MATCH (c:Compound {id: 'Compound::DB00682'}) RETURN count(c) AS 노드수"))

---
## 2-2. 왜 이름이 아니라 `id` 를 키로 삼을까

적재 셀은 `MERGE (n:Compound {id: row.id})` 로 노드를 만들었습니다. 이름이 더 읽기 쉬운데 왜 `id` 를 썼을까요. 데이터에게 직접 물어보겠습니다.

In [ ]:
# 이름이 두 개 이상의 노드에 걸린 경우가 있는지 세어 본다
# n.name 을 그룹핑 키로 두고 센 뒤, WITH 뒤의 WHERE 로 2개 이상인 것만 남긴다
# 레이블들 을 함께 모아 두면 무엇과 무엇이 겹쳤는지 눈으로 볼 수 있다
rows = run_cypher("""
    MATCH (n) WITH n.name AS 이름, collect(labels(n)[0]) AS 레이블들, count(*) AS 노드수
    WHERE 노드수 > 1
    RETURN 이름, 레이블들 ORDER BY 이름
""")
for row in rows:
    print(row)

In [ ]:
# 이름을 키로 삼았다면 여기 나온 쌍이 한 노드로 합쳐졌을 것이다
print("이름이 겹치는 경우:", len(rows), "건")

`Cholecalciferol` 은 **약물**이면서 동시에 **약효분류**입니다. 이 그래프에는 이런 이름이 6건 있습니다.

만약 적재를 `MERGE (n {name: row.name})` 으로 했다면 어떻게 됐을까요. 이 6쌍이 **한 노드로 합쳐졌을** 것입니다. 약물에 붙어야 할 `BINDS` 관계와 약효분류에 붙어야 할 `INCLUDES` 관계가 한 노드에 뒤섞이고, 그때부터 모든 집계가 틀립니다.

**이름은 사람이 읽는 표시이고, 키는 `id` 입니다.** 그래서 제약도 `id` 에 겁니다. 이 원칙은 다음 단원들에서 여러 출처의 데이터를 합칠 때 더 중요해집니다.

## 2-3. 제약은 네 가지다

| 제약 | 무엇을 요구하나 |
|---|---|
| 유일성(uniqueness) | 그 속성 값이 겹치지 않을 것 |
| 존재(existence) | 그 속성이 반드시 있을 것 |
| 키(key) | 유일성과 존재를 한꺼번에 |
| 타입(type) | 그 속성이 정해진 자료형일 것 |

다만 **판(edition)에 따라 만들 수 있는 것이 다릅니다.** 무료판에서는 보통 유일성만 만들어집니다. 표를 외우는 대신 **내 환경에서 무엇이 되는지 직접 걸어 보는** 것이 확실합니다.

In [ ]:
checks = [
    ("유일성", """
        CREATE CONSTRAINT try_unique IF NOT EXISTS
        FOR (s:Symptom) REQUIRE s.name IS UNIQUE
    """),
    ("존재", """
        CREATE CONSTRAINT try_exists IF NOT EXISTS
        FOR (s:Symptom) REQUIRE s.name IS NOT NULL
    """),
    ("타입", """
        CREATE CONSTRAINT try_type IF NOT EXISTS
        FOR (s:Symptom) REQUIRE s.name IS :: STRING
    """),
    ("키", """
        CREATE CONSTRAINT try_key IF NOT EXISTS
        FOR (s:Symptom) REQUIRE (s.name) IS NODE KEY
    """),
]
for label, query in checks:
    try:
        run_cypher(query)
        print(label, "제약: 만들어졌습니다")
        # 확인만 하고 제약 이름(query.split()[2])을 곧바로 지워 뒤 실습에 남기지 않는다
        run_cypher("DROP CONSTRAINT " + query.split()[2] + " IF EXISTS")
    except Exception as error:
        # 여기서는 '어떤 에러든' 못 만든 것으로 본다(판마다 에러 종류가 다르다)
        print(label, "제약: 이 환경에서는 만들 수 없습니다")

### 🖐️ 함께 따라하기: 증상 이름에 UNIQUE 제약 걸기

증상(`Symptom` 415개)의 `name` 은 겹치지 않습니다(앞 셀의 중복 목록에 증상이 없었습니다). 여기에 제약을 걸어 보세요.

1. `CREATE CONSTRAINT symptom_name IF NOT EXISTS FOR (s:Symptom) REQUIRE s.name IS UNIQUE`
2. `SHOW CONSTRAINTS` 로 그 제약이 등록됐는지 확인합니다(이름으로 걸러 출력).
3. 이미 있는 증상 이름 하나를 골라 같은 이름으로 `CREATE` 를 시도하고, `ConstraintError` 를 **그 예외만** 잡아 안내를 출력합니다.
4. 증상 노드 수가 늘지 않았는지 확인합니다.

확인 기준: 3번에서 제약이 막아야 하고, 4번의 개수는 `415` 그대로입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) CREATE CONSTRAINT symptom_name ... REQUIRE s.name IS UNIQUE 를 실행한다
# 2) SHOW CONSTRAINTS YIELD name, labelsOrTypes, properties WHERE name = 'symptom_name' 으로 확인
# 3) 이미 있는 증상 이름(예: 'Fever')으로 CREATE 를 시도하고 except ConstraintError 로 받는다
#    ('Exception' 으로 넓게 받지 말 것. 다른 에러까지 성공으로 삼킨다)
# 4) MATCH (s:Symptom) RETURN count(s) 로 개수가 그대로인지 확인

### ✅ 바로 확인 퀴즈

**1)** 이미 중복이 들어 있는 속성에 UNIQUE 제약을 걸면 어떻게 되나요?

<details><summary>정답 보기</summary>

제약 생성 자체가 **실패**합니다. 규칙을 어기는 데이터가 이미 있으니 규칙을 세울 수 없습니다. 중복을 먼저 정리한 뒤에 걸어야 합니다.

</details>

**2)** `id` 에 제약을 걸어 두면 `CREATE INDEX ... ON (n.id)` 를 따로 만들어야 하나요?

<details><summary>정답 보기</summary>

필요 없습니다. 제약이 **인덱스를 겸합니다.** `SHOW INDEXES` 의 `owningConstraint` 로 확인했듯이, 제약을 만들면 같은 이름의 인덱스가 함께 생깁니다. 따로 만들면 중복이 됩니다.

</details>

**3)** `Cholecalciferol` 처럼 이름이 여러 노드에 걸린 데이터에서, `MERGE (n {name: '이름'})` 로 적재하면 무엇이 잘못되나요?

<details><summary>정답 보기</summary>

서로 다른 것이 **한 노드로 합쳐집니다.** 약물과 약효분류가 한 덩어리가 되어 관계가 뒤섞이고, 그때부터 모든 집계가 조용히 틀립니다. 게다가 레이블이 없으니 인덱스도 못 씁니다. 적재의 키는 **레이블 + 유일한 id** 입니다.

</details>

---
# 3. 추천 랭킹: 무엇으로 줄 세울 것인가

랭킹은 다섯 걸음입니다. **가운데 노드를 세는 정석**을 익히고(3-1), 바뀌는 값을 **파라미터로** 넘기고(3-2), **가운데를 바꾸면 다른 추천**이 나오는 것을 보고(3-3), **그룹마다 상위 N 개**를 뽑고(3-4), 마지막으로 **원점수와 비율이 다른 답을 준다**는 것을 확인합니다(3-5).

## 3-1. 집계 랭킹의 정석: 가운데 노드를 센다

### 왜 필요할까요?
"고혈압에 쓰는 약" 은 앞 시간에 세어 봤습니다. 68개였습니다. 그런데 목록이 68개면 그것대로 쓸모가 없습니다. **무엇을 먼저 볼지** 정해 줘야 합니다.

추천은 결국 **점수를 매겨 줄 세우는 일**입니다. 그래프에서는 그 점수가 대개 "가운데 노드를 몇 개나 공유하는가" 입니다.

### 문법: 집계 랭킹의 정석
```
MATCH (기준)-[:관계]->(가운데)<-[:관계]-(후보)
WHERE 후보 <> 기준                       자기 자신은 뺀다
RETURN 후보.name AS 이름, count(DISTINCT 가운데) AS 점수
ORDER BY 점수 DESC, 이름                 동점은 보조 키로 안정시킨다
LIMIT 5
```
네 가지가 함께 옵니다: **가운데 노드를 세고**, (기준과 후보가 같은 종류라면) **자기 자신을 빼고**, **DISTINCT 로 중복을 없애고**, **보조 정렬 키로 순서를 안정시킨다.**

<img src="images/공유표적_랭킹_패턴.png" width="760">

가운데 노드(유전자)를 함께 가리키는 후보를 세면 그대로 랭킹이 됩니다. 무엇을 가운데에 두느냐가 곧 "무엇이 비슷하다고 볼 것인가"의 정의입니다. 그림의 약물 이름과 숫자는 이 패턴을 보이기 위한 예시이고, 실제 값은 3-3 에서 직접 셉니다.

### 고혈압 약을 무엇으로 줄 세울까

고혈압을 치료하는 약 68개 중, **고혈압과 연관된 유전자를 많이 건드리는** 약을 앞에 놓아 보겠습니다. 가운데 노드는 유전자입니다.

- `(c:Compound)-[:TREATS]->(d:Disease)` : 고혈압 치료약
- `(c)-[:BINDS]->(g:Gene)` : 그 약이 결합하는 유전자
- `(d)-[:ASSOCIATES]->(g)` : 고혈압과 연관됐다고 보고된 유전자

세 조건을 다 만족하는 유전자가 곧 "이 약이 이 병의 기전에 닿는 지점" 입니다.

In [ ]:
# 가운데 g 를 양쪽에서 가리키는 공유 패턴이다. 같은 유전자가 여러 경로로 잡히므로 DISTINCT 를 쓴다
rows = run_cypher("""
    MATCH (c:Compound)-[:TREATS]->(d:Disease {name: 'hypertension'})
    // 약 -> 유전자 <- 병. 가운데 유전자를 약과 병이 함께 가리키고, 그 개수가 곧 점수다
    MATCH (c)-[:BINDS]->(g:Gene)<-[:ASSOCIATES]-(d)
    RETURN c.name AS 약물, count(DISTINCT g) AS 공유유전자수
    ORDER BY 공유유전자수 DESC, 약물 LIMIT 5
""")
for row in rows:
    print(row)

1위 `Carvedilol`(14)이 2위 `Clonidine`(7)의 두 배입니다. 68개를 늘어놓았을 때는 보이지 않던 것이 점수를 붙이자 드러났습니다.

> 이 점수가 "이 약이 제일 좋다"는 뜻은 **아닙니다.** "이 데이터에서 이 병의 기전에 가장 많이 닿는 약이다" 라는 뜻입니다. 추천 점수를 만들 때는 **그 점수가 무엇을 센 것인지**를 항상 말로 설명할 수 있어야 합니다. 설명할 수 없는 점수는 쓸 수 없습니다.

## 3-2. 바뀌는 값은 파라미터로 넘긴다

질병 이름을 바꿔 가며 쓰려면 쿼리 문자열을 매번 새로 만들어야 할까요. 아닙니다. `$이름` 자리표시자를 두고 `run_cypher(쿼리, 이름=값)` 으로 넘깁니다.

- 쿼리 문자열이 **한 벌**이라 데이터베이스가 계획을 재활용해 빠릅니다.
- 값에 따옴표가 들어가도 안전합니다(문자열을 이어 붙이면 쿼리가 깨지거나 엉뚱하게 실행됩니다).

In [ ]:
# 같은 쿼리를 함수로 감싸 두면 질병만 바꿔 가며 부를 수 있다
def rank_drugs(disease_name, top=3):
    """그 질병의 치료약을 '질병 연관 유전자 공유 수' 로 줄 세워 상위 top 개를 돌려준다."""
    # $disease·$top 자리표시자로 값을 넘긴다. 문자열을 이어 붙이지 않는다
    return run_cypher("""
        MATCH (c:Compound)-[:TREATS]->(d:Disease {name: $disease})
        // 약 -> 유전자 <- 병. 가운데 유전자를 약과 병이 함께 가리키고, 그 개수가 곧 점수다
        MATCH (c)-[:BINDS]->(g:Gene)<-[:ASSOCIATES]-(d)
        RETURN c.name AS 약물, count(DISTINCT g) AS 공유유전자수
        ORDER BY 공유유전자수 DESC, 약물 LIMIT $top
    """, disease=disease_name, top=top)

# 질병 세 개를 차례로 넣어 본다. 쿼리 문자열은 한 벌 그대로이고 값만 바뀐다
for disease in ["hypertension", "breast cancer", "type 2 diabetes mellitus"]:
    print(disease, "->", rank_drugs(disease))

## 3-3. 가운데를 바꾸면 다른 추천이 나온다

가운데 노드를 바꾸면 다른 추천이 나옵니다. 이번에는 **약물 두 개가 같은 유전자에 결합하는지**를 셉니다. 표적이 겹친다는 것은 작용하는 자리가 비슷하다는 뜻입니다.

약을 개발할 때 "이미 있는 약 중에 비슷한 것이 없나"를 먼저 훑는 일이 있습니다. 그 첫 단계가 이 쿼리입니다.

In [ ]:
# WHERE other <> c 로 자기 자신을 뺀다. 빼지 않으면 언제나 자기가 1위다
rows = run_cypher("""
    // 기준 약 -> 유전자 <- 다른 약. 같은 표적을 공유하는 약을 찾는다
    MATCH (c:Compound {name: $drug})-[:BINDS]->(g:Gene)<-[:BINDS]-(other:Compound)
    WHERE other <> c
    RETURN other.name AS 약물, count(DISTINCT g) AS 공유표적수
    ORDER BY 공유표적수 DESC, 약물 LIMIT 5
""", drug="Simvastatin")   # 약 이름도 이어 붙이지 않고 파라미터로 넘긴다
for row in rows:
    print(row)

1위 `Atorvastatin`(17), 2위 `Lovastatin`(16). 이 셋은 모두 이름이 `-statin` 으로 끝납니다. **같은 계열의 고지혈증 약**입니다. 우리는 계열 정보를 준 적이 없습니다. 표적을 공유한다는 것만 세었는데 계열이 드러났습니다.

> 이것이 그래프로 추천을 만드는 힘입니다. "비슷하다"를 사람이 라벨로 붙여 주지 않아도, **무엇을 공유하는지 세는 것만으로** 비슷함이 나옵니다.

### 🖐️ 함께 따라하기: 증상이 비슷한 질병 찾기

데모는 **유전자**를 가운데 두었습니다. 따라하기는 **증상**을 가운데 두어 질병끼리 견줍니다.

`similar_disease(disease_name, top=5)` 함수를 만들어, 그 질병과 `PRESENTS` 로 이어진 증상을 가장 많이 공유하는 다른 질병 상위 `top` 개를 돌려주세요.

- 패턴: `(d:Disease {name: $disease})-[:PRESENTS]->(s:Symptom)<-[:PRESENTS]-(other:Disease)`
- 자기 자신은 뺍니다.
- 별칭은 `질병`·`공유증상수`, 정렬은 `공유증상수 DESC, 질병`.
- 값은 **파라미터로** 넘깁니다.

확인 기준: `"breast cancer"` 로 부르면 1위가 `head and neck cancer`(16)입니다.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) def similar_disease(disease_name, top=5): 로 함수를 만든다 (한 줄 docstring 포함)
# 2) 공유 패턴을 MATCH 로 쓴다: (질병)-[:PRESENTS]->(증상)<-[:PRESENTS]-(다른질병)
# 3) WHERE other <> d 로 자기 자신을 빼고, count(DISTINCT s) 로 공유 증상 수를 센다
# 4) ORDER BY 공유증상수 DESC, 질병 LIMIT $top, 값은 $disease·$top 파라미터로 넘긴다
# 5) 'breast cancer' 와 'hypertension' 두 가지로 불러 결과를 출력한다

### ✅ 바로 확인 퀴즈

**1)** `WHERE other <> c` 를 빼면 결과가 어떻게 되나요?

<details><summary>정답 보기</summary>

자기 자신이 1위로 올라옵니다. 자기 표적은 100% 겹치기 때문입니다. 공유 패턴 랭킹에서는 **자기 자신을 빼는 줄이 거의 언제나 필요합니다.**

</details>

**2)** `count(DISTINCT g)` 대신 `count(g)` 를 쓰면 무엇이 달라지나요?

<details><summary>정답 보기</summary>

같은 유전자에 이르는 경로가 여러 개면 그 유전자를 여러 번 셉니다. 점수가 부풀려져 순위가 뒤집힐 수 있습니다. **노드 개수를 세는 자리에는 `DISTINCT`** 입니다.

</details>

**3)** 질병 이름을 문자열로 이어 붙여 `"... {name: '" + 이름 + "'}"` 로 쓰면 무엇이 문제인가요?

<details><summary>정답 보기</summary>

이름에 따옴표가 들어 있으면 쿼리가 깨지거나 엉뚱한 쿼리가 실행됩니다. 그리고 쿼리 문자열이 매번 달라져 데이터베이스가 실행계획을 재활용하지 못합니다. **바뀌는 값은 파라미터로** 넘깁니다.

</details>

---
## 3-4. 그룹마다 상위 N 개

### 왜 필요할까요?
지금까지는 **전체에서** 상위 5개를 뽑았습니다. 실무에서 훨씬 자주 나오는 물음은 "**분류마다** 대표를 세 개씩" 입니다. 카테고리별 인기 상품, 지역별 매출 1위, 약효분류별 대표 약물 같은 것들입니다.

`ORDER BY` 는 상위 몇 개를 남기고 나머지를 버리는 것이 아니라 **줄을 세울 뿐**입니다. 그 줄 세운 순서 그대로 `collect` 로 접으면, 리스트 앞쪽이 곧 상위입니다. 거기서 `[0..3]` 으로 잘라 내면 그룹마다 상위 3개가 됩니다. 앞 시간에 배운 두 가지를 **순서대로 붙이는 것**이 전부입니다.

### 문법: 정렬한 뒤에 접는다
```
MATCH ...
WITH 그룹, 항목, 점수식 AS 점수
ORDER BY 점수 DESC, 항목.name          먼저 줄을 세우고
WITH 그룹, collect(항목.name)[0..3] AS 대표   그 순서대로 접어 앞 3개만 남긴다
RETURN ...
```
**`ORDER BY` 가 `collect` 보다 위에 있어야** 합니다. 순서가 뒤바뀌면 아무 순서로 담깁니다.

<img src="images/그룹별_상위N.png" width="760">

`LIMIT` 은 **전체를 한 줄로 세워** 앞에서 자르고, `collect(...)[0..N]` 은 **묶음마다** 따로 자릅니다. "분류마다 세 개씩" 은 뒤쪽입니다.

In [ ]:
# 약효분류마다 '표적 유전자를 가장 많이 건드리는 약물' 상위 3개를 뽑는다
# 1) 약물마다 표적 수를 센다  2) 그 점수로 줄을 세운다  3) 줄 세운 순서대로 접어 앞 3개만 남긴다
rows = run_cypher("""
    // 약효분류 -> 소속 약 -> 그 약의 표적. 두 홉을 이어 분류마다 모은다
    MATCH (p:PharmacologicClass)-[:INCLUDES]->(c:Compound)-[:BINDS]->(g:Gene)
    WITH p, c, count(DISTINCT g) AS 표적수
    ORDER BY 표적수 DESC, c.name
    WITH p, count(c) AS 약물수, collect(c.name)[0..3] AS 대표약물
    RETURN p.name AS 약효분류, 약물수, 대표약물
    ORDER BY 약물수 DESC, 약효분류 LIMIT 3
""")
for row in rows:
    print(row)

1위 분류 `Corticosteroid Hormone Receptor Agonists` 에는 **표적이 있는 약물**이 22개 있고, 그중 표적을 가장 많이 건드리는 셋이 `['Dexamethasone', 'Hydrocortisone', 'Fluticasone Propionate']` 입니다. `LIMIT` 은 **전체에서** 자르지만, `collect(...)[0..3]` 은 **그룹마다** 자릅니다.

In [ ]:
# ORDER BY 를 빼면 어떻게 되나: collect 는 담기는 순서를 보장하지 않는다
# 이 셀의 출력은 실행할 때마다 달라질 수 있다. '아무 순서' 라는 것이 이 셀의 요점이다
rows = run_cypher("""
    // 이 분류에 든 약 -> 그 약의 표적. 분류 하나만 잡아 두 홉을 간다
    MATCH (p:PharmacologicClass {name: $cls})-[:INCLUDES]->(c:Compound)-[:BINDS]->(g:Gene)
    WITH p, c, count(DISTINCT g) AS 표적수
    WITH p, collect(c.name)[0..3] AS 대표약물
    RETURN 대표약물
""", cls="Corticosteroid Hormone Receptor Agonists")
print("정렬 없이 접으면:", rows[0]["대표약물"])

In [ ]:
# 1위를 손으로 적지 않는다. 정렬해서 뽑아야 데이터가 바뀌어도 거짓이 되지 않는다
top = run_cypher("""
    // 이 분류에 든 약 -> 그 약의 표적. 분류 하나만 잡아 두 홉을 간다
    MATCH (:PharmacologicClass {name: $cls})-[:INCLUDES]->(c:Compound)-[:BINDS]->(g:Gene)
    WITH c, count(DISTINCT g) AS 표적수
    RETURN c.name AS 약물 ORDER BY 표적수 DESC, 약물 LIMIT 1
""", cls="Corticosteroid Hormone Receptor Agonists")[0]["약물"]
print("표적수 1위는:", top)

### ✅ 바로 확인 퀴즈

**1)** `LIMIT 3` 과 `collect(...)[0..3]` 은 무엇이 다른가요?

<details><summary>정답 보기</summary>

`LIMIT 3` 은 **결과 전체**에서 앞 3행만 남깁니다. `collect(...)[0..3]` 은 **그룹마다** 모은 리스트에서 앞 3개를 남깁니다. "분류마다 3개씩" 을 물었다면 뒤쪽이 답입니다.

</details>

**2)** `ORDER BY` 를 `collect` 아래에 두면 어떻게 되나요?

<details><summary>정답 보기</summary>

이미 접힌 뒤라 **리스트 안의 순서는 바뀌지 않습니다.** 아래의 `ORDER BY` 는 그룹(행)의 순서만 바꿉니다. 리스트 안을 정렬하려면 **접기 전에** 줄을 세워야 합니다.

</details>

---
## 3-5. 원점수와 비율은 다른 답을 준다

### 왜 필요할까요?
지금까지 만든 점수는 전부 **몇 개나 공유하는가**(원점수)였습니다. 이 점수에는 알고 써야 할 성질이 하나 있습니다. **원래 연결이 많은 쪽이 유리하다**는 것입니다. 표적이 100개인 약은 무엇과 견주어도 겹치는 것이 많습니다.

3-1 의 고혈압 랭킹에 **그 약의 전체 표적 수**를 나란히 놓아 보겠습니다.

In [ ]:
# 3-1 의 랭킹에 '그 약이 원래 몇 개를 건드리는가' 를 덧붙인다
# 적중률 = 공유 유전자 수 / 그 약의 전체 표적 수. 정수끼리 나누면 버림이 되므로 toFloat 로 실수로 만든다
rows = run_cypher("""
    MATCH (c:Compound)-[:TREATS]->(d:Disease {name: $disease})
    // 약 -> 유전자 <- 병. 가운데 유전자를 약과 병이 함께 가리키고, 그 개수가 곧 점수다
    MATCH (c)-[:BINDS]->(g:Gene)<-[:ASSOCIATES]-(d)
    WITH c, count(DISTINCT g) AS 공유유전자수
    // 앞의 c 를 그대로 다시 쓴다. 이번엔 공유와 상관없이 그 약의 표적을 전부 센다(분모)
    MATCH (c)-[:BINDS]->(t:Gene)
    WITH c, 공유유전자수, count(DISTINCT t) AS 전체표적수
    RETURN c.name AS 약물, 공유유전자수, 전체표적수,
           round(toFloat(공유유전자수) / 전체표적수, 3) AS 적중률
    ORDER BY 공유유전자수 DESC, 약물 LIMIT 5
""", disease="hypertension")
for row in rows:
    print(row)

원점수 1위는 `Carvedilol`(공유 14 · 전체 표적 26 · 적중률 0.538)입니다. 그런데 이 표에서 **적중률이 가장 높은 약은 `Captopril`**(공유 6 · 전체 표적 11 · 적중률 0.545)입니다. 원점수로는 3위인 약입니다.

**두 점수가 다른 질문에 답하고 있습니다.**

| 점수 | 무엇을 묻는가 |
|---|---|
| 원점수(공유 개수) | 이 병의 기전에 **가장 많이** 닿는 약은? |
| 비율(적중률) | 하는 일 중에 이 병과 상관있는 몫이 **가장 큰** 약은? |

둘 중 무엇이 옳은 것이 아니라, **묻는 것이 다릅니다.** 어느 쪽을 쓸지는 그 추천을 무엇에 쓸지가 정합니다.

### 그러면 비율로 줄 세우면 되나요

그렇게 간단하지 않습니다. 비율만으로 정렬하면 **다른 방향으로 무너집니다.** 3-3 의 `Simvastatin` 유사약 찾기를 비율로 바꿔 보겠습니다.

In [ ]:
# 3-3 의 유사약 랭킹을 '공유 표적 수' 가 아니라 '겹침 비율' 로 줄 세운다
# 겹침비율 = 공유 표적 수 / 그 후보의 전체 표적 수
rows = run_cypher("""
    // 기준 약 -> 유전자 <- 다른 약. 같은 표적을 공유하는 약을 찾는다
    MATCH (c:Compound {name: $drug})-[:BINDS]->(g:Gene)<-[:BINDS]-(other:Compound)
    WHERE other <> c
    WITH other, count(DISTINCT g) AS 공유표적수
    // other 를 그대로 다시 써서 그 후보의 전체 표적을 센다(분모)
    MATCH (other)-[:BINDS]->(t:Gene)
    WITH other, 공유표적수, count(DISTINCT t) AS 전체표적수
    RETURN other.name AS 약물, 공유표적수, 전체표적수,
           round(toFloat(공유표적수) / 전체표적수, 3) AS 겹침비율
    ORDER BY 겹침비율 DESC, 공유표적수 DESC, 약물 LIMIT 5
""", drug="Simvastatin")
for row in rows:
    print(row)

1위가 `Artemether` 입니다. 겹침비율 1.0 이니 완벽해 보입니다. 그런데 이 약의 전체 표적은 **6개뿐**입니다. 그 6개가 마침 모두 Simvastatin(표적 18개)의 표적 안에 들어 있었을 뿐입니다. **표적이 적을수록 비율이 쉽게 1.0 이 됩니다.**

3-3 에서 원점수로 뽑았을 때 앞에 왔던 `-statin` 계열은 이 줄에서 사라졌습니다. 비율은 인기 편향을 없애 주는 대신 **반대쪽으로 치우쳤습니다.**

### 고치는 법: 너무 작은 후보를 걸러 낸다

가장 단순하고 흔한 처방은 **최소 크기 조건**입니다. "전체 표적이 10개 이상인 후보만" 처럼 한 줄을 더합니다. 집계한 뒤의 조건이니 `WITH` 뒤의 `WHERE` 자리입니다(앞 교안의 HAVING 그 자리).

In [ ]:
# 같은 비율 랭킹에 '전체 표적이 최소 몇 개는 되는 후보만' 이라는 조건을 더한다
# 집계한 뒤에야 알 수 있는 값이 조건이므로 WITH 뒤의 WHERE 에 쓴다
rows = run_cypher("""
    // 기준 약 -> 유전자 <- 다른 약. 같은 표적을 공유하는 약을 찾는다
    MATCH (c:Compound {name: $drug})-[:BINDS]->(g:Gene)<-[:BINDS]-(other:Compound)
    WHERE other <> c
    WITH other, count(DISTINCT g) AS 공유표적수
    // other 를 그대로 다시 써서 그 후보의 전체 표적을 센다(분모)
    MATCH (other)-[:BINDS]->(t:Gene)
    WITH other, 공유표적수, count(DISTINCT t) AS 전체표적수
    WHERE 전체표적수 >= $최소표적
    RETURN other.name AS 약물, 공유표적수, 전체표적수,
           round(toFloat(공유표적수) / 전체표적수, 3) AS 겹침비율
    ORDER BY 겹침비율 DESC, 공유표적수 DESC, 약물 LIMIT 5
""", drug="Simvastatin", 최소표적=10)
for row in rows:
    print(row)

조건 한 줄로 1위가 `Artemether` 에서 `Indinavir` 으로 바뀌었고, 3-3 에서 원점수 2위였던 `Lovastatin`(공유 16 / 19 = 0.842)이 다시 올라왔습니다.

> `10` 이라는 수에 근거가 있느냐고 물으면, 정직한 답은 **"이 데이터를 보고 정한 값"** 입니다. 이런 문턱값은 데이터마다 다시 정해야 하고, **얼마로 정했는지 적어 두어야** 합니다. 적어 두지 않은 문턱값은 나중에 아무도 설명하지 못합니다.

**오늘 세 가지 점수를 만들었습니다.**

| 점수 | 치우치는 방향 | 언제 쓰나 |
|---|---|---|
| 원점수(공유 개수) | 연결이 많은 쪽이 유리 | "가장 많이 닿는 것" 을 물을 때 |
| 비율(공유 / 전체) | 연결이 적은 쪽이 유리 | "몫이 큰 것" 을 물을 때 |
| 비율 + 최소 조건 | 둘 사이 | 대개 실무에서 쓰는 모양 |

> 뒤 단원에서 배울 **노드 유사도**(Node Similarity)는 이 비율을 더 다듬은 것입니다. 양쪽을 함께 보아 `공유 / (A 전체 + B 전체 - 공유)` 로 계산합니다. 지금 손으로 만들어 본 이 고민이 그 알고리즘이 푸는 문제입니다.

### 🖐️ 함께 따라하기: 증상 유사도를 비율로 바꿔 보기

3-3 따라하기에서 만든 `similar_disease` 는 **공유 증상 수**로 줄 세웠습니다. 이번에는 비율판 `similar_disease_ratio(disease_name, top=5, min_symptoms=5)` 를 만드세요.

- 같은 공유 패턴으로 `공유증상수` 를 센 뒤, `MATCH (other)-[:PRESENTS]->(t:Symptom)` 으로 **그 질병의 전체 증상 수**를 셉니다.
- `WHERE 전체증상수 >= $min_symptoms` 로 너무 작은 후보를 거릅니다.
- `round(toFloat(공유증상수) / 전체증상수, 3) AS 겹침비율` 을 내고 `ORDER BY 겹침비율 DESC, 공유증상수 DESC, 질병` 으로 정렬합니다.
- `"breast cancer"` 로 **원점수판과 비율판을 나란히 출력**해 1위가 같은지 다른지 보세요.

확인 기준: 두 판의 1위가 서로 다르게 나옵니다. `toFloat` 를 빼면 겹침비율이 전부 `0` 이 됩니다(정수 나눗셈 버림). 그것도 한 번 확인해 보세요.

In [ ]:
# 🖐️ 함께 따라하기 (아래 순서대로 직접 작성해 보세요)
# 1) def similar_disease_ratio(disease_name, top=5, min_symptoms=5): (한 줄 docstring)
# 2) 공유 패턴으로 공유증상수를 센 뒤, WITH 로 넘겨 그 질병의 전체 증상 수를 다시 센다
# 3) WHERE 전체증상수 >= $min_symptoms 로 너무 작은 후보를 거른다
# 4) round(toFloat(공유증상수) / 전체증상수, 3) 으로 비율을 내고 정렬한다
# 5) 'breast cancer' 로 similar_disease 와 나란히 불러 1위를 견준다

### ✅ 바로 확인 퀴즈

**1)** `round(공유표적수 / 전체표적수, 3)` 처럼 `toFloat` 없이 쓰면 어떻게 되나요?

<details><summary>정답 보기</summary>

정수끼리 나누면 **소수점 아래가 버려집니다.** 공유가 전체보다 작으니 결과는 거의 전부 `0` 이 되고, 모든 후보가 동점이 되어 순위가 뜻을 잃습니다. 에러는 나지 않습니다. **한쪽을 `toFloat` 로 감싸세요.**

</details>

**2)** 비율 랭킹에서 `WHERE 전체표적수 >= 10` 은 왜 `WITH` 뒤에 와야 하나요?

<details><summary>정답 보기</summary>

`전체표적수` 는 **집계해서 만든 이름**이라 `MATCH` 바로 뒤의 `WHERE` 에는 아직 존재하지 않습니다. 앞 교안에서 배운 HAVING 자리와 같습니다.

</details>

**3)** "이 약과 가장 비슷한 약" 을 추천할 때 원점수와 비율 중 무엇을 쓰겠습니까?

<details><summary>정답 보기</summary>

정해진 답이 없는 것이 이 질문의 요점입니다. **무엇에 쓸 추천인지**가 정합니다. 다만 어느 쪽을 고르든 **그 점수가 어느 쪽으로 치우치는지**를 알고 골라야 하고, 비율을 쓴다면 **최소 크기 조건을 함께** 두어야 합니다. 그리고 그 선택을 리포트에 적어 두어야 합니다.

</details>

---
## 🚀 응용 클론코딩: 질병 하나에 대한 추천 리포트

오늘 배운 것을 한 함수에 모읍니다. 질병 이름을 받아, 그 병의 치료약을 점수 순으로 **약효분류까지 붙여** 돌려주는 리포트입니다.

**요구사항**

- 함수 이름은 `drug_report(disease_name, top=3)`, 한 줄 docstring 을 답니다.
- 받은 질병 이름으로 그 병의 치료약을 잡습니다.
- 약마다 그 병과 함께 가리키는 유전자가 몇 개인지 셉니다. **공유 유전자가 하나도 없는 약도 목록에서 빠지면 안 됩니다.**
- 이어서 약마다 약효분류를 붙여 목록으로 모읍니다(분류가 없는 약도 있습니다).
- 공유 유전자 수 내림차순, 동점이면 약 이름순으로 위에서부터 `top` 개만 냅니다.
- 질병 이름과 개수는 쿼리에 박지 말고 **파라미터로** 넘깁니다.
- 결과 칸 이름은 `약물`·`공유유전자수`·`약효분류` 로 합니다.

확인 기준: `"hypertension"` 으로 부르면 1위가 `Carvedilol`(공유유전자수 14)이고, 그 약효분류 목록의 길이는 2 입니다. 3위 `Captopril` 은 약효분류가 비어 있는 목록(`[]`)으로 나옵니다.

In [ ]:
# 🚀 응용 (아래 순서대로 직접 작성해 보세요)
# 1) def drug_report(disease_name, top=3): 로 함수를 만든다 (한 줄 docstring)
# 2) MATCH 로 치료약을 잡고, OPTIONAL MATCH 로 공유 유전자를 잇는다
# 3) WITH c, count(DISTINCT g) AS 공유유전자수  로 1차 집계
# 4) OPTIONAL MATCH 로 약효분류를 잇고 collect(p.name) 으로 모은다
# 5) ORDER BY 공유유전자수 DESC, 약물 LIMIT $top, 값은 $disease·$top 파라미터로
# 6) 'hypertension' 으로 불러 한 줄씩 출력한다

> 리포트를 사람에게 보여 줄 때는 **점수가 무엇을 센 것인지**와 **이 데이터가 무엇인지**를 함께 적어야 합니다. 여기 나온 순위는 "2016년까지 그렇게 보고된 관계를 센 결과" 이지 "임상적으로 그 순서가 낫다" 는 뜻이 아닙니다.

---
## 이번 강의 정리

| 하고 싶은 일 | 쓰는 것 |
|---|---|
| 조회를 빠르게 한다 | `CREATE INDEX 이름 IF NOT EXISTS FOR (n:레이블) ON (n.속성)` |
| 문자열 부분일치를 빠르게 한다 | `CREATE TEXT INDEX ...`(RANGE 로는 `CONTAINS`·`ENDS WITH` 를 못 짚는다) |
| 관계 속성에 인덱스를 건다 | `CREATE INDEX ... FOR ()-[r:타입]-() ON (r.속성)` |
| 어떤 방법으로 실행됐는지 본다 | `PROFILE 쿼리`(실행하며 비용까지) / `EXPLAIN 쿼리`(계획만) |
| 중복을 막는다 | `CREATE CONSTRAINT ... REQUIRE n.속성 IS UNIQUE` |
| 목록을 확인한다 | `SHOW INDEXES` / `SHOW CONSTRAINTS` |
| 추천 점수를 만든다 | 가운데 노드를 `count(DISTINCT ...)` 로 센다 |
| 그룹마다 상위 N 개를 뽑는다 | `ORDER BY` 로 줄 세운 뒤 `collect(...)[0..N]` |
| 점수를 비율로 바꾼다 | `round(toFloat(공유) / 전체, 3)` + 최소 크기 조건 |
| 순위를 안정시킨다 | `ORDER BY 점수 DESC, 보조키` |
| 바뀌는 값을 넘긴다 | `$파라미터` + `run_cypher(쿼리, 이름=값)` |

**오늘 실측으로 확인한 것**

| 무엇을 | 일을 많이 한 쪽 | 적게 한 쪽 | 차이 |
|---|---|---|---|
| 유전자 하나 찾기(dbHits) | 26,228 인덱스 없이 | 3 인덱스로 | 8,743배 |
| 관계 200줄 적재(dbHits) | 12,436,099 레이블 뺌 | 4,099 레이블 적음 | 3,034배 |
| 이름에 조각이 든 것 찾기(dbHits) | 13,114 RANGE | 9 TEXT | 1,457배 |

**인덱스를 못 쓰게 되는 세 가지**

1. **레이블을 뺐다** (`AllNodesScan`, dbHits 31,082). 인덱스는 레이블마다 따로 걸립니다.
2. **속성을 함수로 감쌌다** (`NodeByLabelScan`, dbHits 26,228). 인덱스는 저장된 값에만 걸립니다.
3. **인덱스 종류가 맞지 않다** (`CONTAINS`·`ENDS WITH` 에 RANGE 인덱스). TEXT 인덱스가 필요합니다.

- **제약은 인덱스를 겸합니다.** 유일해야 하는 키에는 인덱스가 아니라 제약을 겁니다.
- **키는 이름이 아니라 `id`** 입니다. 이 그래프에도 이름이 겹치는 노드가 6건 있습니다.
- 추천은 **가운데 노드를 세어 줄 세우는 일**입니다. 그 점수가 무엇을 센 것인지 말로 설명할 수 있어야 합니다.
- **원점수는 연결이 많은 쪽에, 비율은 연결이 적은 쪽에 치우칩니다.** 어느 쪽을 골랐는지와 최소 크기 조건을 얼마로 두었는지를 리포트에 적으세요.

## ⏭️ 예고: 다음 시간

지금까지는 준비 셀이 만들어 준 그래프 위에서 조회만 했습니다. **다음 단원**에서는 그래프를 **직접 적재**합니다. 노드를 먼저 넣고 관계를 나중에 잇는 순서가 왜 그래야 하는지를 작은 조각으로 익힌 뒤, CSV 를 읽어 수만 줄을 옮깁니다.

오늘 배운 인덱스·제약이 그때 바로 쓰입니다. 중복이 이미 있는 상태에서는 제약을 걸려는 것조차 실패한다는 것을 다음 단원에서 직접 보게 됩니다.

수고하셨습니다!